# Lesson 3.6 — How Do I Chunk PDFs and Scanned Documents?

**Companion notebook for the blog post.**

This notebook covers the hardest, most under-appreciated problem in RAG engineering: getting clean text out of real documents before you even start chunking.

### What you'll learn:
- Why PDFs are structurally different from what you expect
- How to choose between PyMuPDF, pdfplumber, and PyPDF2
- How to extract tables and preserve their structure for LLMs
- How to handle scanned PDFs with OCR (Tesseract, AWS Textract, Google Document AI)

### Document used:
We use the **WHO Guideline on Healthy School Food Environments** throughout — the same PDF from earlier lessons, already in `data/who_document.pdf`.

---
## 📦 Step 0: Install Dependencies

Run once. Restart the kernel after installation if needed.

In [ ]:
# Uncomment and run once
# !pip install PyMuPDF pdfplumber pypdf2 pytesseract Pillow langchain langchain-community

In [ ]:
import os
os.environ["USE_TF"] = "0"

import urllib.request
from pathlib import Path

PDF_PATH        = "data/who_document.pdf"        # single-column WHO guideline
ARXIV_PDF_PATH  = "data/arxiv_2512.04123.pdf"    # two-column arxiv paper ("Measuring Agents in Production")

if not Path(PDF_PATH).exists():
    print(f"⚠️  WHO PDF not found at '{PDF_PATH}'.")
    print("    Run the Lesson 2.3 notebook first to download it.")
else:
    print(f"✅ WHO PDF found: {PDF_PATH} ({Path(PDF_PATH).stat().st_size / 1024:.1f} KB)")

# Download the two-column arxiv paper if not already present
if not Path(ARXIV_PDF_PATH).exists():
    print(f"\nDownloading arxiv paper for multi-column demo...")
    req = urllib.request.Request(
        "https://arxiv.org/pdf/2512.04123",
        headers={"User-Agent": "Mozilla/5.0"}
    )
    with urllib.request.urlopen(req) as r, open(ARXIV_PDF_PATH, "wb") as f:
        f.write(r.read())
    print(f"✅ Downloaded: {ARXIV_PDF_PATH}")
else:
    print(f"✅ Arxiv PDF found: {ARXIV_PDF_PATH} ({Path(ARXIV_PDF_PATH).stat().st_size / 1024:.1f} KB)")

---
---
## Part 1: Handling PDFs

### Why PDFs Are So Hard

A PDF does **not** contain text the way you think it does.

A PDF is essentially a set of instructions for *drawing* things on a page:

```
Move to position (72, 680)  → Draw the glyph for "H"
Move to position (78, 680)  → Draw the glyph for "e"
Move to position (84, 680)  → Draw the glyph for "l"
```

There are **no paragraphs, sentences, or headings** — just coordinates and glyphs. Your PDF viewer reconstructs the reading experience visually, but the underlying data is a scattered mess.

This means text extraction is inherently **lossy**. You're reverse-engineering structure from visual layout, and things go wrong constantly — especially with multi-column layouts.

---
## 🔧 Library 1: PyMuPDF (fitz)

**Best default choice** — fast, handles most document layouts well, supports multiple extraction modes:

| Mode | What it gives you |
|---|---|
| `"text"` | Plain text, fast |
| `"blocks"` | Text grouped into layout blocks |
| `"dict"` | Full structure: fonts, sizes, bounding boxes |

### The Multi-Column Problem

The `"text"` mode reads blocks top-to-bottom across the **full page width**. On a two-column academic paper this silently interleaves both columns — producing text that looks readable but is actually garbled for an LLM.

The cell below demonstrates this using **["Measuring Agents in Production"](https://arxiv.org/abs/2512.04123)** — a real arxiv paper with a classic two-column layout. Watch what happens to page 4 under naive vs. layout-aware extraction.

In [ ]:
# ============================================
# PyMuPDF (fitz) — Basic "text" mode
# Works well on single-column documents like our WHO PDF.
# Breaks on multi-column layouts — see the next cell for why.
# ============================================
import fitz

def extract_with_pymupdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc):
        text = page.get_text("text")   # "text" | "blocks" | "dict"
        pages.append({"page": page_num + 1, "text": text})
    doc.close()
    return pages


pages_basic = extract_with_pymupdf(PDF_PATH)
print(f"✅ Extracted {len(pages_basic)} pages from WHO PDF")
print(f"\n📄 Page 1 preview ({len(pages_basic[0]['text'])} chars):")
print("-" * 60)
print(pages_basic[0]["text"][:500])
print("-" * 60)
print("\n💡 This looks fine — the WHO PDF is single-column.")
print("   Run the next cell to see what happens on a two-column paper.")

In [ ]:
# ============================================================
# The Multi-Column Problem — a real arxiv paper
# Paper: "Measuring Agents in Production" (arxiv 2512.04123)
# This is a classic two-column academic layout.
# ============================================================
import fitz

def block_text(block):
    """Extract readable text from a PyMuPDF dict block."""
    return " ".join(
        span["text"]
        for line in block["lines"]
        for span in line["spans"]
    ).strip()


PAGE_NUM  = 3          # Page 4 — dense two-column body text with a figure
PAGE_MID  = 300        # x-coordinate that separates left and right columns


doc = fitz.open(ARXIV_PDF_PATH)
page = doc[PAGE_NUM]
all_blocks = [b for b in page.get_text("dict")["blocks"] if b["type"] == 0]

# ── NAIVE: sort purely by Y (top→bottom), ignoring columns ──────────────────
# This is what happens when a library doesn't understand multi-column layout.
# Blocks from both columns are interleaved based on their vertical position.
naive_blocks = sorted(all_blocks, key=lambda b: b["bbox"][1])
naive_text   = "\n".join(block_text(b) for b in naive_blocks if block_text(b))

# ── LAYOUT-AWARE: left column first, then right column ──────────────────────
# Split blocks at the page midpoint, sort each column by Y independently.
left_col  = sorted([b for b in all_blocks if b["bbox"][0] <  PAGE_MID], key=lambda b: b["bbox"][1])
right_col = sorted([b for b in all_blocks if b["bbox"][0] >= PAGE_MID], key=lambda b: b["bbox"][1])
layout_text = "\n".join(block_text(b) for b in (left_col + right_col) if block_text(b))

doc.close()

# ── Print side-by-side ───────────────────────────────────────────────────────
PREVIEW = 600

print("❌ NAIVE extraction  (blocks sorted by Y only — columns interleaved)")
print("=" * 70)
print(naive_text[:PREVIEW])

print(f"\n\n✅ LAYOUT-AWARE extraction  (left column → right column)")
print("=" * 70)
print(layout_text[:PREVIEW])

print(f"\n\n📊 Stats for page {PAGE_NUM + 1}:")
print(f"   Total text blocks : {len(all_blocks)}")
print(f"   Left column blocks: {len(left_col)}")
print(f"   Right col blocks  : {len(right_col)}")

---
## 🔧 Library 2: pdfplumber

**Best for tables** — its superpower is detecting and extracting tabular data from PDFs (invoices, financial reports, research papers with data tables).

**Why convert tables to Markdown?**  
LLMs understand Markdown tables surprisingly well. If you concatenate table cells into a flat string, you lose the row/column relationships:

```
❌ Flat:     "Drug  Resistance%  Region  Cases"
✅ Markdown: | Drug | Resistance% | Region | Cases |
```

The Markdown version preserves structure that both embeddings and LLMs can work with.

In [ ]:
# ============================================
# pdfplumber — Best for tables
# ============================================
import pdfplumber  # pip install pdfplumber


def format_table_as_text(header, rows):
    """Convert a table into LLM-friendly Markdown."""
    lines = []
    lines.append("| " + " | ".join(str(h or "") for h in header) + " |")
    lines.append("|" + "|".join("---" for _ in header) + "|")
    for row in rows:
        lines.append("| " + " | ".join(str(cell or "") for cell in row) + " |")
    return "\n".join(lines)


def extract_with_pdfplumber(pdf_path):
    results = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            text = page.extract_text()

            # Extract tables — pdfplumber's superpower
            tables = page.extract_tables()

            formatted_tables = []
            for table in tables:
                if table and len(table) > 1:
                    header = table[0]
                    rows = table[1:]
                    formatted_tables.append(format_table_as_text(header, rows))

            results.append({
                "page": page_num + 1,
                "text": text,
                "tables": formatted_tables,
                "table_count": len(formatted_tables),
            })
    return results


pages_plumber = extract_with_pdfplumber(PDF_PATH)
pages_with_tables = [p for p in pages_plumber if p["table_count"] > 0]

print(f"✅ Extracted {len(pages_plumber)} pages")
print(f"📊 Pages containing tables: {len(pages_with_tables)}")

if pages_with_tables:
    example = pages_with_tables[0]
    print(f"\n--- Table found on page {example['page']} ---")
    print(example["tables"][0][:600])
else:
    print("\n(No tables detected — this PDF uses image-based tables or text layout)")

---
## 🔍 Side-by-Side Comparison

Same page, different libraries — see how extraction quality differs.

In [ ]:
# Compare PyMuPDF basic vs pdfplumber on page 5
PAGE = 4  # 0-indexed → page 5

pymupdf_text  = pages_basic[PAGE]["text"][:400]
plumber_text  = pages_plumber[PAGE]["text"] or ""
plumber_text  = plumber_text[:400]

print(f"{'='*55}")
print(f"  PyMuPDF — Page {PAGE+1}")
print(f"{'='*55}")
print(pymupdf_text)

print(f"\n{'='*55}")
print(f"  pdfplumber — Page {PAGE+1}")
print(f"{'='*55}")
print(plumber_text)

# 🎛️ EXPERIMENT: Try different page numbers to compare extraction quality

---
## 📋 Library Decision Guide

| Library | Speed | Table support | Multi-column | Best for |
|---|---|---|---|---|
| **PyMuPDF** | ⚡ Fast | Basic | Good | Most RAG pipelines (default choice) |
| **pdfplumber** | Moderate | Excellent | Moderate | Invoices, reports, financial docs |
| **PyPDF2** | Fast | Poor | Poor | Simple docs, zero external dependencies |

> **Rule of thumb:** Start with PyMuPDF. Switch to pdfplumber only if tables are critical to your use case.

---
---
## Part 2: Scanned PDFs — When There's No Text At All

Some PDFs are just **images of pages**. There's literally no text data to extract — just pixels. You can see the text in a viewer but try to select it and... nothing.

These require **OCR (Optical Character Recognition)** — converting images of text into actual text.

```
Scanned PDF  →  Image (pixels)  →  OCR Engine  →  Text  →  Chunks
```

The strategy below auto-detects whether a page has real text or needs OCR — so you can handle mixed PDFs (some pages scanned, some digital).

---
## 🔧 Option 1: Tesseract (Free, Local)

**Pros:** Free, runs locally, no data leaves your machine, supports 100+ languages.  
**Cons:** Accuracy varies with image quality. Struggles with complex layouts and handwriting.  
**Best for:** Clear, simple scanned documents when privacy/compliance requirements prevent cloud use.

**Setup:**
```bash
# macOS
brew install tesseract

# Linux
apt install tesseract-ocr
```

In [ ]:
# ============================================
# Tesseract OCR — Free, local, good enough
# ============================================
import fitz
from PIL import Image
import pytesseract  # pip install pytesseract
import io


def ocr_scanned_pdf(pdf_path, dpi=300):
    """
    Hybrid extractor: uses direct text extraction where text exists,
    falls back to Tesseract OCR for scanned (image-only) pages.

    Args:
        pdf_path: path to the PDF file
        dpi: render resolution for OCR — higher = better accuracy, slower
             300 is the recommended minimum for good OCR results

    Returns:
        list of dicts with page number, text, and method used
    """
    doc = fitz.open(pdf_path)
    pages = []

    for page_num, page in enumerate(doc):
        text = page.get_text("text").strip()

        if len(text) > 50:
            # Page has real extractable text — use it directly
            pages.append({
                "page": page_num + 1,
                "text": text,
                "method": "direct"
            })
        else:
            # Page is likely scanned — render it as an image and OCR it
            pix = page.get_pixmap(dpi=dpi)           # Higher DPI = better OCR
            img = Image.open(io.BytesIO(pix.tobytes("png")))

            # --psm 3: fully automatic page segmentation (good default)
            # Add lang='eng+fra' for multilingual docs
            ocr_text = pytesseract.image_to_string(img, config="--psm 3")
            pages.append({
                "page": page_num + 1,
                "text": ocr_text,
                "method": "ocr"
            })

    doc.close()
    return pages


# Run on our WHO PDF (mostly digital, so most pages will use "direct")
ocr_pages = ocr_scanned_pdf(PDF_PATH)

direct_count = sum(1 for p in ocr_pages if p["method"] == "direct")
ocr_count    = sum(1 for p in ocr_pages if p["method"] == "ocr")

print(f"✅ Processed {len(ocr_pages)} pages")
print(f"   Direct text extraction: {direct_count} pages")
print(f"   OCR (Tesseract):        {ocr_count} pages")
print(f"\n📄 Page 1 ({ocr_pages[0]['method']}) preview:")
print("-" * 60)
print(ocr_pages[0]["text"][:400])
print("-" * 60)

### 💡 Tesseract Pro Tips

1. **Always use 300 DPI or higher** when rendering pages to images — lower resolution kills accuracy
2. **Pre-process images**: deskew, remove noise, increase contrast before passing to Tesseract
3. **Use `--psm` flags** to tell Tesseract about the layout:
   - `--psm 3`: Fully automatic page segmentation (default)
   - `--psm 6`: Assume a single uniform block of text
   - `--psm 4`: Assume a single column of text
4. **Specify language**: `pytesseract.image_to_string(img, lang='eng+fra')` for multilingual documents

---
## ☁️ Option 2: AWS Textract (Cloud, Powerful)

**Pros:** Excellent accuracy, understands tables/forms/key-value pairs, returns structured data.  
**Cons:** $1.50 per 1,000 pages (basic) / $15 per 1,000 pages (tables+forms). Data goes to AWS.  
**Best for:** Business documents, forms, invoices, reports with tables — when accuracy matters more than cost.

> **Requirements:** AWS account + `boto3` + IAM credentials configured (`aws configure`)

In [ ]:
# ============================================
# AWS Textract — When you need table extraction from scans
# pip install boto3
# aws configure  (set your credentials first)
# ============================================
# import boto3
#
# def ocr_with_textract(pdf_path):
#     """Use AWS Textract for high-quality OCR with table detection."""
#     client = boto3.client('textract')
#
#     with open(pdf_path, 'rb') as f:
#         response = client.analyze_document(
#             Document={'Bytes': f.read()},
#             FeatureTypes=['TABLES', 'FORMS']  # Detect tables AND forms
#         )
#
#     # Extract text lines
#     text_blocks = []
#     for block in response['Blocks']:
#         if block['BlockType'] == 'LINE':
#             text_blocks.append(block['Text'])
#
#     return '\n'.join(text_blocks)
#
# result = ocr_with_textract(PDF_PATH)
# print(result[:500])

print("ℹ️  AWS Textract code is commented out — requires AWS credentials.")
print("   Uncomment and run after setting up: aws configure")

---
## ☁️ Option 3: Google Document AI (Cloud, State-of-the-Art)

**Pros:** State-of-the-art accuracy, specialized processors for invoices/receipts/contracts, handles handwriting.  
**Cons:** Costs money, data goes to Google Cloud, more complex setup.  
**Best for:** When you need the highest possible accuracy, especially for specialized document types.

> **Requirements:** Google Cloud project + Document AI API enabled + processor created in the console

In [ ]:
# ============================================
# Google Document AI — State-of-the-art OCR
# pip install google-cloud-documentai
# gcloud auth application-default login
# ============================================
# from google.cloud import documentai_v1 as documentai
#
# client = documentai.DocumentProcessorServiceClient()
#
# # Create a processor first in the Google Cloud Console
# processor_name = "projects/YOUR_PROJECT/locations/us/processors/YOUR_PROCESSOR"
#
# with open(PDF_PATH, "rb") as f:
#     raw_document = documentai.RawDocument(
#         content=f.read(),
#         mime_type="application/pdf"
#     )
#
# request = documentai.ProcessRequest(
#     name=processor_name,
#     raw_document=raw_document
# )
#
# result = client.process_document(request=request)
# document = result.document
#
# # Full extracted text
# print(document.text[:500])
#
# # Structured entities (available with specialized processors)
# for entity in document.entities:
#     print(f"{entity.type_}: {entity.mention_text}")

print("ℹ️  Google Document AI code is commented out — requires GCP credentials.")
print("   Uncomment after: gcloud auth application-default login")

---
## 📋 OCR Engine Decision Guide

| Engine | Cost | Privacy | Accuracy | Table support | Best for |
|---|---|---|---|---|---|
| **Tesseract** | Free | Local only | Good | Poor | Simple scans, privacy-sensitive docs |
| **AWS Textract** | ~$1.50–15/1k pages | AWS cloud | Excellent | Excellent | Business forms, invoices, reports |
| **Google Doc AI** | Pay per page | Google cloud | State-of-art | Excellent | Complex docs, handwriting, specialized types |

### How to decide:
```
Can data leave your infrastructure?
    ├── No  → Tesseract (free, local)
    └── Yes → Do you need table/form extraction?
                  ├── Yes → AWS Textract or Google Document AI
                  └── No  → Tesseract is probably good enough
```

---
## 🔁 Putting It All Together: A Production-Ready Extractor

Here's a single function that picks the right strategy automatically based on what it finds in each page.

In [ ]:
import fitz
import pdfplumber
from PIL import Image
import pytesseract
import io
from langchain.schema import Document


def smart_pdf_extractor(pdf_path, ocr_threshold=50, extract_tables=True, dpi=300):
    """
    Production-ready PDF extractor that:
      - Uses PyMuPDF for fast text extraction on digital pages
      - Falls back to Tesseract OCR for scanned / image-only pages
      - Optionally detects and formats tables as Markdown (via pdfplumber)

    Args:
        pdf_path:       path to the PDF
        ocr_threshold:  if a page has fewer than this many characters, treat as scanned
        extract_tables: whether to detect tables with pdfplumber
        dpi:            render DPI for OCR pages (300 recommended)

    Returns:
        list of LangChain Document objects, one per page
    """
    fitz_doc = fitz.open(pdf_path)
    documents = []

    # Build a pdfplumber page index for table extraction
    plumber_pages = {}
    if extract_tables:
        with pdfplumber.open(pdf_path) as plumber_pdf:
            for i, page in enumerate(plumber_pdf.pages):
                tables = page.extract_tables()
                if tables:
                    plumber_pages[i] = tables

    for page_num, page in enumerate(fitz_doc):
        text = page.get_text("text").strip()
        method = "direct"

        if len(text) < ocr_threshold:
            # Scanned page — OCR it
            pix = page.get_pixmap(dpi=dpi)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            text = pytesseract.image_to_string(img, config="--psm 3")
            method = "ocr"

        # Append Markdown tables found on this page
        table_text = ""
        if page_num in plumber_pages:
            for table in plumber_pages[page_num]:
                if table and len(table) > 1:
                    header = table[0]
                    rows = table[1:]
                    md = format_table_as_text(header, rows)
                    table_text += f"\n\n[TABLE]\n{md}\n[/TABLE]\n"

        full_text = text + table_text

        documents.append(Document(
            page_content=full_text,
            metadata={
                "source": str(pdf_path),
                "page": page_num + 1,
                "extraction_method": method,
                "has_tables": page_num in plumber_pages,
            }
        ))

    fitz_doc.close()
    return documents


# Run the smart extractor on our WHO PDF
docs = smart_pdf_extractor(PDF_PATH)

direct = sum(1 for d in docs if d.metadata["extraction_method"] == "direct")
ocr    = sum(1 for d in docs if d.metadata["extraction_method"] == "ocr")
tables = sum(1 for d in docs if d.metadata["has_tables"])

print(f"✅ Smart extractor done: {len(docs)} pages")
print(f"   Direct text: {direct} pages")
print(f"   OCR pages:   {ocr} pages")
print(f"   With tables: {tables} pages")
print(f"\n📄 Sample document (page 2):")
print("-" * 60)
print(docs[1].page_content[:400])
print("-" * 60)
print(f"\n📎 Metadata: {docs[1].metadata}")

---
## 🧪 Exercises: Your Turn!

1. **Compare extraction modes**: Change `page.get_text("text")` to `page.get_text("blocks")` in the PyMuPDF cell. How does the output differ?
2. **Force OCR on a digital page**: In `smart_pdf_extractor`, set `ocr_threshold=9999` so every page goes through Tesseract. Compare accuracy with direct extraction.
3. **Bring your own PDF**: Replace `PDF_PATH` with a scanned document and observe how many pages fall back to OCR.
4. **Table formatting**: Find a PDF with a table, extract it with pdfplumber, and compare the Markdown output with the raw flattened string. Which would an LLM answer better from?